# Day 11：文件操作与常用标准库（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 2 周**：第 2 周：面向对象、工程化与命令行 AI 助手实战  
> **GitLab 工单**：`STARMIND-011` ｜ **开发分支**：`feat/day11-files-stdlib-kb-scanner`  
> **配套 HTML5 课件**：`courseware/day11/index.html` ｜ **独立版本快照**：`versions/day11/`

---

## 🎙️ 老陈开场旁白

今天咱们把 Python「自带电池（Batteries Included）」的威力彻底释放出来！星辰数科知识库目录里躺着上百份 `.txt` 和 `.md` 制度文档，林产品想做一个上线前的数据摸底（STARMIND-011）：自动递归扫描整个文件夹，用正则表达式提取所有文档里的邮箱、工单号和核心关键词出现频次，打上 `datetime` 时间戳，最终同时导出 JSON 摘要报告和给运营看的 CSV 报表！今天你学会了 `pathlib`、`csv`、`datetime` 和 `re` 正则这四把标准库神兵，以后做 RAG 批量文档导入轻轻松松。

### 承上启下
- **昨日基线**：昨天咱们搭好了 StarMind 自定义异常体系和主备容错调度器，工程骨架已经立起来了。
- **明日预告**：明天（Day 12）激动人心的时刻到了——咱们学习 HTTP 协议与 API 调用，亲手对接 DeepSeek / 通义千问协议写出第一版命令行 AI 问答小程序！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day11'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-011）

在搭建 RAG 知识库前，运营与算法团队需要对存量散落的本地 Markdown/TXT 文档进行批量内容扫描、敏感信息抽检与高频业务词统计，并生成可供 Excel 打开的 CSV 统计表。

**今日迭代目标：**
- 掌握 `with open(..., encoding='utf-8')`、`pathlib.Path` 跨平台目录遍历与文件读写。
- 熟练使用 `csv.DictWriter/DictReader`、`datetime` 时间戳格式化与 `re` 正则表达式。
- 交付批量文档扫描、关键词频次统计、实体抽取与 CSV/JSON 双格式报表导出工具。

**验收标准：**
- 能递归扫描指定目录下所有 `.txt` / `.md` 文件并正确统计字符数与指定关键词命中次数。
- 能用正则准确提取文档中出现的邮箱地址与 `STARMIND-xxx` 工单号。
- 能将统计结果无乱码导出为 CSV 与 JSON 报告，并在隔离 `tmp_path` 下通过 pytest。

---

## 📚 课堂核心技术笔记

### 1. 1. pathlib.Path：现代 Python 文件路径唯一推荐写法

用 `/` 运算符优雅拼接路径：`base = Path('data') / 'kb' / 'guide.md'`；常用属性与方法：`.exists()`、`.suffix`（扩展名）、`.stem`（不含扩展名的文件名）、`.read_text(encoding='utf-8')`、`.rglob('*.md')` 递归搜索。

> **⚠️ 老陈避坑**：尽量避免手写反斜杠 `'C:\new\test.txt'`（`\n` 和 `\t` 会被当成转义符），用 `Path` 或原始字符串 `r'...'` 彻底免疫转义坑。

### 2. 2. CSV 读写与 Excel 中文防乱码技巧

写 CSV 务必用 `open(path, 'w', encoding='utf-8-sig', newline='')` 搭配 `csv.DictWriter`。其中 `newline=''` 交给 csv 模块自己控制换行，避免 Windows 下每两行之间多出一条空白行；`utf-8-sig` 带 BOM 头，微软 Excel 直接双击打开不乱码。

> **⚠️ 老陈避坑**：用 `csv.DictReader` 读回数据时，所有列默认都是字符串 `str`，数值列记得手动 `int()` / `float()` 转换。

### 3. 3. re 正则表达式与 datetime 时区意识

正则字符串一律加 `r` 前缀（原始字符串），高频函数：`re.findall` 提取全部、`re.sub` 替换脱敏、`re.compile` 预编译提速。时间处理在 Python 3.11+ 推荐 `datetime.now(timezone.utc)`（带时区感知），避免使用已废弃的 `datetime.utcnow()`。

> **⚠️ 老陈避坑**：正则表达式里的 `.*` 默认是**贪婪匹配**，提取两个标签之间的内容时请用非贪婪 `.*?`，否则会把整篇文章吞成一大段！


## 💻 实操步骤 1：步骤一：编写预编译正则实体抽取与关键词频次分析器

- **目标文件**：`versions/day11/src/day11_regex_extractor.py`
- **老陈代码旁白**：先用 `re.compile` 预编译好星辰数科常用的邮箱、工单号和关键词统计函数，单文档分析逻辑全部做成纯函数。


In [ ]:
"""
STARMIND-011 模块一：预编译正则实体抽取与关键词频次统计器。
演示 re.compile、re.findall、大小写不敏感匹配与边界词频统计。
"""

import re
from typing import Dict, List, Any

# 预编译高频业务正则表达式，批量扫描时避免重复编译开销
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
TICKET_PATTERN = re.compile(r"STARMIND-\d{3,4}", re.IGNORECASE)


def extract_entities(text: str) -> Dict[str, List[str]]:
    """从文档正文中抽取去重排序后的邮箱地址与星辰工单号。"""
    emails = sorted(set(EMAIL_PATTERN.findall(text)))
    # 工单号统一标准化为大写
    tickets = sorted({t.upper() for t in TICKET_PATTERN.findall(text)})
    return {"emails": emails, "tickets": tickets}


def count_keywords(text: str, keywords: List[str]) -> Dict[str, int]:
    """统计指定关键词列表在文本中出现的总次数（大小写不敏感）。"""
    lowered = text.lower()
    counts: Dict[str, int] = {}
    for kw in keywords:
        clean_kw = kw.strip()
        if not clean_kw:
            continue
        counts[clean_kw] = lowered.count(clean_kw.lower())
    return counts


def analyze_document_text(name: str, text: str, keywords: List[str]) -> Dict[str, Any]:
    """汇总单篇文档的字数、行数、实体及关键词命中明细。"""
    entities = extract_entities(text)
    kw_counts = count_keywords(text, keywords)
    return {
        "file_name": name,
        "char_count": len(text),
        "line_count": len(text.splitlines()),
        "emails": entities["emails"],
        "tickets": entities["tickets"],
        "keyword_counts": kw_counts,
        "total_keyword_hits": sum(kw_counts.values()),
    }


if __name__ == "__main__":
    sample = "需求 STARMIND-011 由 lin@starmind.cn 提出，关联 starmind-005，重点关注 RAG 与 Python 性能优化。"
    print(analyze_document_text("demo.md", sample, ["RAG", "Python", "Agent"]))


### 🔍 步骤 1 代码深度解析

1. `re.compile` 放在模块顶层，程序启动只编译一次，后续扫描一万份文档速度极快。
2. 工单号正则加了 `re.IGNORECASE`，然后统一 `.upper()` 归一化，防止文档里有人手滑写成 `starmind-005` 漏统计。

**预期运行输出参考：**
```text
{'file_name': 'demo.md', 'char_count': 54, 'line_count': 1, 'emails': ['lin@starmind.cn'], 'tickets': ['STARMIND-005', 'STARMIND-011'], 'keyword_counts': {'RAG': 1, 'Python': 1, 'Agent': 0}, 'total_keyword_hits': 2}
```


## 💻 实操步骤 2：步骤二：实现批量文档递归扫描器与 CSV / JSON 双格式报表导出

- **目标文件**：`versions/day11/src/day11_doc_scanner.py`
- **老陈代码旁白**：现在把 `pathlib`、`csv`、`json`、`datetime` 串起来，实现目录批量扫描并一键生成运营 CSV 和系统 JSON 报告。


In [ ]:
"""
STARMIND-011 模块二：知识库批量文档扫描与 CSV/JSON 双报表导出器。
演示 pathlib.Path 递归扫描、datetime 时区时间戳、csv.DictWriter(utf-8-sig) 与 json.dump。
"""

import csv
import json
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Any, List, Tuple

from day11_regex_extractor import analyze_document_text

SUPPORTED_SUFFIXES: Tuple[str, ...] = (".txt", ".md")


def scan_directory(root_dir: Path, keywords: List[str]) -> Dict[str, Any]:
    """递归扫描目录下所有 .txt / .md 文件，生成完整统计报告。"""
    if not root_dir.exists() or not root_dir.is_dir():
        raise FileNotFoundError(f"扫描目录不存在: {root_dir}")

    doc_reports: List[Dict[str, Any]] = []
    global_keyword_totals: Dict[str, int] = {kw: 0 for kw in keywords}

    # 使用 pathlib.rglob 递归遍历并按相对路径排序，确保结果稳定
    matched_files = sorted(
        p for p in root_dir.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES
    )

    for file_path in matched_files:
        text = file_path.read_text(encoding="utf-8")
        rel_name = file_path.relative_to(root_dir).as_posix()
        report = analyze_document_text(rel_name, text, keywords)
        doc_reports.append(report)
        for kw, cnt in report["keyword_counts"].items():
            global_keyword_totals[kw] = global_keyword_totals.get(kw, 0) + cnt

    return {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "scanned_dir": root_dir.as_posix(),
        "total_files": len(doc_reports),
        "global_keyword_totals": global_keyword_totals,
        "documents": doc_reports,
    }


def export_reports(summary: Dict[str, Any], output_dir: Path) -> Dict[str, Path]:
    """将扫描结果同时导出为 JSON 完整报告与 CSV 运营明细表。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    json_path = output_dir / "kb_scan_report.json"
    csv_path = output_dir / "kb_scan_summary.csv"

    # 1. 导出 JSON
    with json_path.open("w", encoding="utf-8") as jf:
        json.dump(summary, jf, ensure_ascii=False, indent=2)

    # 2. 导出 CSV（utf-8-sig 确保 Excel 打开中文不乱码，newline='' 防多余空行）
    fieldnames = ["file_name", "char_count", "line_count", "total_keyword_hits", "tickets", "emails"]
    with csv_path.open("w", encoding="utf-8-sig", newline="") as cf:
        writer = csv.DictWriter(cf, fieldnames=fieldnames)
        writer.writeheader()
        for doc in summary["documents"]:
            writer.writerow(
                {
                    "file_name": doc["file_name"],
                    "char_count": doc["char_count"],
                    "line_count": doc["line_count"],
                    "total_keyword_hits": doc["total_keyword_hits"],
                    "tickets": ";".join(doc["tickets"]),
                    "emails": ";".join(doc["emails"]),
                }
            )

    return {"json": json_path, "csv": csv_path}


if __name__ == "__main__":
    # 使用临时目录自包含演示，运行完自动清理不留垃圾文件
    with tempfile.TemporaryDirectory() as tmp:
        kb_dir = Path(tmp) / "kb"
        (kb_dir / "tech").mkdir(parents=True)
        (kb_dir / "readme.md").write_text("星辰智脑 Python RAG 指南，联系人 chen@starmind.cn", encoding="utf-8")
        (kb_dir / "tech" / "api.txt").write_text("工单 STARMIND-011：Python 与 FastAPI 接口规范", encoding="utf-8")

        summary = scan_directory(kb_dir, ["Python", "RAG", "FastAPI"])
        paths = export_reports(summary, Path(tmp) / "out")
        print(f"扫描完成: {summary['total_files']} 份文档 | 全局词频: {summary['global_keyword_totals']}")
        print("CSV 预览:\n" + paths["csv"].read_text(encoding="utf-8-sig").strip())


### 🔍 步骤 2 代码深度解析

1. `file_path.relative_to(root_dir).as_posix()` 无论在 Windows 还是 Linux 上都输出统一的 `/` 相对路径，报表格式跨平台一致。
2. `__main__` 演示块使用 `tempfile.TemporaryDirectory()`，跑完自动销毁临时目录，极度干净。

**预期运行输出参考：**
```text
扫描完成: 2 份文档 | 全局词频: {'Python': 2, 'RAG': 1, 'FastAPI': 1}
CSV 预览:
file_name,char_count,line_count,total_keyword_hits,tickets,emails
readme.md,34,1,2,,chen@starmind.cn
tech/api.txt,34,1,2,STARMIND-011,
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day11/tests/test_day11.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-011 自动化单元测试。"""

import csv
import json
import sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day11_regex_extractor import extract_entities, count_keywords
from day11_doc_scanner import scan_directory, export_reports


def test_regex_extraction_and_keywords():
    text = "联系 hr@starmind.cn 或 dev@starmind.cn，参考 starmind-008 和 STARMIND-011。Python 很好用，python！"
    ent = extract_entities(text)
    assert ent["emails"] == ["dev@starmind.cn", "hr@starmind.cn"]
    assert ent["tickets"] == ["STARMIND-008", "STARMIND-011"]
    assert count_keywords(text, ["Python", "Java"]) == {"Python": 2, "Java": 0}


def test_scan_directory_and_export_csv_json(tmp_path: Path):
    kb = tmp_path / "docs"
    (kb / "sub").mkdir(parents=True)
    (kb / "a.md").write_text("星辰智脑 RAG 方案 STARMIND-001", encoding="utf-8")
    (kb / "sub" / "b.txt").write_text("RAG 与 Agent 结合 dev@starmind.cn", encoding="utf-8")
    (kb / "ignore.bin").write_bytes(b"\x00\x01")  # 非文本文件应自动忽略

    summary = scan_directory(kb, ["RAG", "Agent"])
    assert summary["total_files"] == 2
    assert summary["global_keyword_totals"] == {"RAG": 2, "Agent": 1}

    out = export_reports(summary, tmp_path / "reports")
    loaded_json = json.loads(out["json"].read_text(encoding="utf-8"))
    assert loaded_json["total_files"] == 2

    with out["csv"].open("r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == 2
    assert rows[0]["file_name"] == "a.md"
    assert rows[1]["file_name"] == "sub/b.txt"

    with pytest.raises(FileNotFoundError):
        scan_directory(tmp_path / "not_exist", ["RAG"])

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：正则表达式批量脱敏手机号

**题目要求：**
编写 `mask_phones_in_text(text: str) -> str`，使用 `re.sub` 将任意长文本中出现的所有 11 位中国大陆手机号（1 开头）自动替换为前 3 后 4 脱敏格式（如 `138****8899`）。

**思路提示：** 使用捕获组 `r'(1[3-9]\d)\d{4}(\d{4})'` 配合替换串 `r'\1****\2'`。


In [ ]:
"""Day 11 课后题 1：正则捕获组批量脱敏文本中的手机号。"""

import re

PHONE_RE = re.compile(r"(?<!\d)(1[3-9]\d)\d{4}(\d{4})(?!\d)")


def mask_phones_in_text(text: str) -> str:
    """将文本中所有独立的 11 位手机号替换为中间四位掩码。"""
    return PHONE_RE.sub(r"\1****\2", text)


if __name__ == "__main__":
    raw = "客户手机 13857108899，备用 18612345678，订单号 13857108899001 不应误伤。"
    print(mask_phones_in_text(raw))


**答案解析与评分标准：**

利用负向环视 `(?<!\d)` 与 `(?!\d)` 精准匹配独立 11 位手机号，不会误伤 14 位数字订单号！


### 作业 2（企业实战）：企业实战：日志文件按日期归档命名器

**题目要求：**
编写 `build_log_path(base_dir: Path, service_name: str, dt: datetime) -> Path`，返回形如 `base_dir/2026/05/starmind_20260511.log` 的分层归档路径。

**思路提示：** 用 `dt.strftime('%Y')`、`dt.strftime('%m')` 和 `dt.strftime('%Y%m%d')` 拼接 `Path`。


In [ ]:
"""Day 11 课后题 2：按年月分目录的日志归档路径生成器。"""

from datetime import datetime, timezone
from pathlib import Path


def build_log_path(base_dir: Path, service_name: str, dt: datetime) -> Path:
    """生成按 年/月 分层的标准日志文件路径。"""
    return base_dir / dt.strftime("%Y") / dt.strftime("%m") / f"{service_name}_{dt.strftime('%Y%m%d')}.log"


if __name__ == "__main__":
    now = datetime(2026, 5, 11, 10, 30, tzinfo=timezone.utc)
    print(build_log_path(Path("logs"), "starmind", now).as_posix())


**答案解析与评分标准：**

按年月分桶存放日志是运维标配，避免单目录塞进几十万个文件导致 `ls` 卡死。


### 作业 3（架构进阶）：架构进阶：从大模型回复中提取所有 Markdown 代码块

**题目要求：**
大模型经常在回答中夹杂多个 ```` ```python ... ``` ```` 代码块。编写 `extract_code_blocks(markdown_text: str) -> list[dict]`，用非贪婪正则提取每个代码块的语言标识 `lang` 和纯代码正文 `code`。

**思路提示：** 使用 `re.compile(r'```(\w*)\n(.*?)```', re.DOTALL)` 配合 `findall`。


In [ ]:
"""Day 11 课后题 3：大模型 Markdown 回复代码块提取器。"""

import re
from typing import List, Dict

CODE_BLOCK_RE = re.compile(r"```(\w*)\n(.*?)```", re.DOTALL)


def extract_code_blocks(markdown_text: str) -> List[Dict[str, str]]:
    """提取 Markdown 中所有围栏代码块及其语言类型。"""
    blocks: List[Dict[str, str]] = []
    for lang, code in CODE_BLOCK_RE.findall(markdown_text):
        blocks.append({"lang": lang.lower() or "text", "code": code.strip()})
    return blocks


if __name__ == "__main__":
    reply = "示例如下：\n```python\nprint('hello')\n```\n配置：\n```json\n{\"ok\": true}\n```"
    print(extract_code_blocks(reply))


**答案解析与评分标准：**

本题直接可用于代码执行 Agent（Code Interpreter）自动抽取大模型生成的 Python 代码并送入沙盒运行！注意 `re.DOTALL` 让 `.` 能跨行匹配，非贪婪 `.*?` 防止把两个代码块合并成一个。


---

## 🎯 老陈收尾复盘

记住今天的两条现代 Python 铁律：1）写路径操作无脑用 `pathlib.Path`，彻底告别满屏 `os.path.join` 的斜杠地狱；2）写 CSV 报表在 Windows 下如果怕 Excel 打开中文乱码，把编码设为 `utf-8-sig`（带 BOM 的 UTF-8），运营同学双击打开绝不乱码！明天咱们就要联网调大模型 API 了，保持状态！
